# Camera Discovery — Pipeline-only Profile Comparison

This notebook runs the normal `camera-discovery run` pipeline without harvest input for `fast`, `balanced`, and `full` profiles. It compares candidate counts, validation behavior, trusted/review artifacts, source-row behavior, browser diagnostics, and output summaries.

It does **not** test harvest mode. Use the harvest notebooks for raw URL/media extraction behavior. Full profile can run substantially longer than fast mode.


## Setup

This notebook installs repository code from the configured Git branch and runs the public CLI. It does not patch source files from the notebook.

Default behavior clones the `dev` branch. In Colab, restart the runtime after dependency installation if package imports behave unexpectedly.


In [ ]:
# Repository setup for Google Colab / notebook execution.
# Change REPO_BRANCH or REPO_URL if testing a fork/PR branch.
REPO_BRANCH = "dev"
REPO_URL = "https://github.com/dshipley71/camera-discovery.git"
REPO_DIR = "/content/camera-discovery"

from pathlib import Path
repo_dir = Path(REPO_DIR)
if not repo_dir.exists():
    !git clone -b "{REPO_BRANCH}" "{REPO_URL}" "{REPO_DIR}"
else:
    print(f"Repository already exists at {repo_dir}. Keeping existing checkout.")
%cd {REPO_DIR}
%pip install -e .[cloakbrowser] --no-build-isolation


## Credentials

In [ ]:
# Ollama Cloud / LLM credential setup.
# This avoids printing secrets. Configure OLLAMA_API_KEY in Colab: left sidebar > Secrets.
import os

try:
    from google.colab import userdata  # type: ignore
    OLLAMA_API_KEY = userdata.get('OLLAMA_API_KEY')
except Exception:
    OLLAMA_API_KEY = os.environ.get('OLLAMA_API_KEY')

if OLLAMA_API_KEY:
    os.environ['OLLAMA_API_KEY'] = OLLAMA_API_KEY
    os.environ.setdefault('CAMERA_DISCOVERY_LLM_PROVIDER', 'ollama-cloud')
    os.environ.setdefault('CAMERA_DISCOVERY_LLM_MODEL', 'gemma3:27b-cloud')
    print('Loaded OLLAMA_API_KEY from Colab userdata/environment')
else:
    print('OLLAMA_API_KEY not found. LLM-backed stages may fail unless another provider is configured.')

# Keep these visible so output records the provider/model, but never print the key.
print('LLM provider:', os.environ.get('CAMERA_DISCOVERY_LLM_PROVIDER', '(default from config)'))
print('LLM model:', os.environ.get('CAMERA_DISCOVERY_LLM_MODEL', '(default from config)'))


## Smoke tests

In [ ]:
# CLI and public import smoke tests.
import subprocess, sys

def run_cmd(cmd, *, env=None, check=True):
    print("\n$", " ".join(str(part) for part in cmd))
    result = subprocess.run([str(part) for part in cmd], env=env, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(map(str, cmd))}")
    return result

run_cmd(['camera-discovery', '--help'])
run_cmd(['camera-discovery', 'run', '--help'])
run_cmd(['camera-discovery', 'harvest-urls', '--help'])

from camera_discovery.services.discovery_engine import CandidateDiscoveryEngine
from camera_discovery.services.harvest_engine import CameraUrlHarvestEngine
import camera_discovery.cli
print('camera-discovery imports OK')

import camera_discovery
from camera_discovery.utils import geojson_viewer
print('camera_discovery package:', camera_discovery.__file__)
print('geojson_viewer module:', geojson_viewer.__file__)
print('target bbox overlay helper available:', hasattr(geojson_viewer, 'load_target_geometry_overlays'))


## Notebook-only helper functions

In [ ]:
# Notebook-only inspection helpers. These intentionally live in the notebook, not src/.
from __future__ import annotations

import json
import os
import shutil
import subprocess
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse


def read_json(path):
    path = Path(path)
    if not path.exists():
        print(f"Missing: {path}")
        return None
    return json.loads(path.read_text(encoding='utf-8'))


def iter_jsonl(path, limit=None):
    path = Path(path)
    if not path.exists():
        return
    with path.open('r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            if limit is not None and idx >= limit:
                break
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)


def count_jsonl(path):
    path = Path(path)
    if not path.exists():
        return 0
    with path.open('r', encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())


def top_hosts(path, url_field='url', limit=15):
    counts = Counter()
    for row in iter_jsonl(path):
        url = row.get(url_field) or row.get('stream_url') or row.get('source_url') or ''
        host = urlparse(url).netloc.casefold() or '(missing-host)'
        counts[host] += 1
    return counts.most_common(limit)


def media_counts(path):
    counts = Counter()
    for row in iter_jsonl(path):
        counts[row.get('media_type') or row.get('camera_type') or '(missing)'] += 1
    return dict(counts)


def scope_counts(path):
    counts = Counter()
    for row in iter_jsonl(path):
        counts[row.get('scope_status') or row.get('properties', {}).get('scope_status') or '(missing)'] += 1
    return dict(counts)


def print_json(path, keys=None):
    data = read_json(path)
    if data is None:
        return None
    if keys:
        data = {key: data.get(key) for key in keys}
    print(json.dumps(data, indent=2, sort_keys=True)[:12000])
    return data


def list_existing(paths):
    for path in paths:
        path = Path(path)
        print(f"{path}: {'exists' if path.exists() else 'missing'}" + (f" ({path.stat().st_size:,} bytes)" if path.exists() and path.is_file() else ''))


def package_output(output_dir, zip_name=None):
    output_dir = Path(output_dir)
    if zip_name is None:
        zip_name = str(output_dir).rstrip('/').replace('/', '_') + '.zip'
    zip_base = Path(zip_name).with_suffix('')
    archive = shutil.make_archive(str(zip_base), 'zip', root_dir=str(output_dir))
    print('Created archive:', archive)
    try:
        from google.colab import files  # type: ignore
        files.download(archive)
    except Exception:
        print('Download helper unavailable outside Colab. Archive remains at:', archive)
    return archive


def run_cli(cmd, *, env_overrides=None, check=True):
    env = os.environ.copy()
    if env_overrides:
        env.update({k: str(v) for k, v in env_overrides.items()})
    return run_cmd(cmd, env=env, check=check)


## Browser backend behavior

These notebooks default to browser capture disabled for structured endpoint / HLS workflows because the useful camera records usually come from static pages and JSON endpoints. Enable browser capture only when testing dynamic pages.

The cells below make the selected backend visible. They do not fake browser success.


In [ ]:
# Browser backend configuration for this notebook.
# For routine HLS/structured-endpoint tests, keep browser capture disabled.
BROWSER_BACKEND = "playwright"  # change to "cloakbrowser" when intentionally testing that backend
DISABLE_BROWSER_CAPTURE_ENV = {"CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE": "false", "CAMERA_DISCOVERY_BROWSER_BACKEND": BROWSER_BACKEND}
print('Browser backend selected:', BROWSER_BACKEND)
print('Browser capture default for this notebook:', DISABLE_BROWSER_CAPTURE_ENV['CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE'])
print('To test dynamic browser capture, remove --disable-browser-capture for harvest and set CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE=true for run.')


## Run pipeline-only profiles

In [ ]:
QUERY = "California traffic cameras"
HTTP_TIMEOUT_SECONDS = 10
PROFILE_DIRS = {
    'fast': Path('runs/pipeline-only-fast'),
    'balanced': Path('runs/pipeline-only-balanced'),
    'full': Path('runs/pipeline-only-full'),
}
RERUN_PIPELINE_PROFILES = False

for profile, output_dir in PROFILE_DIRS.items():
    expected = [output_dir / 'logs' / 'run_summary.json', output_dir / 'logs' / 'candidate_discovery_summary.json']
    if RERUN_PIPELINE_PROFILES and output_dir.exists():
        shutil.rmtree(output_dir)
    if all(path.exists() for path in expected):
        print(f'Skipping {profile}: completion artifacts already exist. Set RERUN_PIPELINE_PROFILES=True to rerun.')
        continue
    print('
' + '=' * 80)
    print('Running profile:', profile)
    print('Output will stream below. Validation has no candidate cap; --http-timeout bounds slow network requests.')
    print('=' * 80)
    !camera-discovery run "{QUERY}"       --profile "{profile}"       --output-dir "{output_dir}"       --browser-backend "{BROWSER_BACKEND}"       --http-timeout "{HTTP_TIMEOUT_SECONDS}"       --progress-style plain


## Compare profile outputs

In [ ]:
# Summarize fast/balanced/full outputs side by side.
for profile, output_dir in PROFILE_DIRS.items():
    print('\n' + '=' * 80)
    print('PROFILE:', profile)
    print('OUTPUT:', output_dir)
    print('=' * 80)
    RUN_DIR = output_dir
    print_json(RUN_DIR / 'logs' / 'run_summary.json')
    print('Candidate discovery summary:')
    print_json(RUN_DIR / 'logs' / 'candidate_discovery_summary.json', keys=[
        'native_discovery_candidates', 'harvest_input_candidates', 'combined_candidates',
        'combined_media_counts', 'scope_counts', 'candidate_priority_counts'
    ])
    print('Validation summary:')
    print_json(RUN_DIR / 'logs' / 'validation_summary.json')
    list_existing([
        RUN_DIR / 'camera_candidates_table.csv',
        RUN_DIR / 'camera.geojson',
        RUN_DIR / 'untrusted_camera_candidates.geojson',
        RUN_DIR / 'review_artifacts.zip',
    ])


## Inspect one selected profile in detail

## Target-resolution bbox/map overlay note

Target-resolution diagnostics now preserve the accepted Nominatim bbox and the effective bbox used downstream. For tiny precise targets, the effective bbox may be padded to the minimum practical extent; the original Nominatim bbox, padding reason, and minimum side length remain in `logs/target_resolution*.json`. Generated maps overlay target bounding boxes as border-only rectangles together with geocoder points and camera coordinate markers.


In [ ]:
# Change SELECTED_PROFILE to inspect one profile with the shared detailed helper.
SELECTED_PROFILE = 'full'
RUN_DIR = PROFILE_DIRS[SELECTED_PROFILE]
print('Selected profile:', SELECTED_PROFILE)
# Inspect pipeline/run outputs.
RUN_DIR = Path(RUN_DIR)
print('Run directory:', RUN_DIR)
list_existing([
    RUN_DIR / 'logs' / 'run_summary.json',
    RUN_DIR / 'logs' / 'run_explanation.json',
    RUN_DIR / 'logs' / 'candidate_discovery_summary.json',
    RUN_DIR / 'logs' / 'candidate_priority_summary.json',
    RUN_DIR / 'logs' / 'validation_summary.json',
    RUN_DIR / 'logs' / 'validation_priority_summary.json',
    RUN_DIR / 'camera_candidates_table.csv',
    RUN_DIR / 'untrusted_camera_candidates.geojson',
    RUN_DIR / 'camera.geojson',
    RUN_DIR / 'map.html',
    RUN_DIR / 'review_artifacts.zip',
])

print('\nRun summary:')
print_json(RUN_DIR / 'logs' / 'run_summary.json')
print('\nCandidate discovery summary:')
print_json(RUN_DIR / 'logs' / 'candidate_discovery_summary.json')
print('\nCandidate priority summary:')
print_json(RUN_DIR / 'logs' / 'candidate_priority_summary.json')
print('\nValidation summary:')
print_json(RUN_DIR / 'logs' / 'validation_summary.json')

# Inspect CSV quickly without pandas.
csv_path = RUN_DIR / 'camera_candidates_table.csv'
if csv_path.exists():
    import csv
    with csv_path.open(newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    print('\nCandidate table rows:', len(rows))
    print('Media counts:', dict(Counter(row.get('camera_type') or row.get('media_type') or '(missing)' for row in rows)))
    print('Scope counts:', dict(Counter(row.get('scope_status') or '(missing)' for row in rows)))
    print('Priority buckets:', dict(Counter(row.get('candidate_priority_bucket') or '(missing)' for row in rows)))
    print('\nFirst 5 candidate table rows:')
    for row in rows[:5]:
        print({k: row.get(k) for k in ['camera_type', 'scope_status', 'candidate_priority_bucket', 'stream_url', 'latitude', 'longitude', 'validation_status'] if k in row})
else:
    print('No candidate table found.')

# GeoJSON feature counts.
for geojson_name in ['camera.geojson', 'untrusted_camera_candidates.geojson']:
    path = RUN_DIR / geojson_name
    data = read_json(path)
    if data:
        features = data.get('features', [])
        print(f"{geojson_name}: {len(features)} features")
        print('Feature scope counts:', dict(Counter((feat.get('properties') or {}).get('scope_status') or '(missing)' for feat in features)))

# Regenerate the map with the currently installed source code so reused run
# directories do not keep an older map.html without target bbox overlays.
from camera_discovery.utils.geojson_viewer import load_target_geometry_overlays, write_embedded_camera_map

target_overlays = load_target_geometry_overlays(RUN_DIR)
print('
Target bbox/point overlays discovered:', len(target_overlays))
if target_overlays:
    print('First target overlay:', {k: target_overlays[0].get(k) for k in ['target_label', 'bbox', 'effective_bbox', 'nominatim_bbox', 'lat', 'lon', 'geometry_source', 'bbox_padding_applied']})
    regenerated_map = write_embedded_camera_map(RUN_DIR, output_name='map.html')
    print('Regenerated map with target overlays:', regenerated_map)
    print('
Camera map status:')
    print_json(RUN_DIR / 'logs' / 'camera_map_status.json')
else:
    print('WARNING: no target-resolution bbox/point overlays found in logs/target_resolution*.json; map will show camera points only.')


## Package outputs

In [ ]:
# Optional: package all pipeline-only profile outputs.
for profile, output_dir in PROFILE_DIRS.items():
    if output_dir.exists():
        package_output(output_dir, zip_name=f'pipeline_only_{profile}.zip')


## Nominatim target geometry hierarchy inspection

This cell verifies that target maps prefer Nominatim polygon/multipolygon boundaries, fall back to the Nominatim rectangular bbox, and only use a generic padded bbox when no usable Nominatim geometry exists.


In [ ]:
# NOMINATIM_GEOMETRY_HIERARCHY_INSPECTION
from pathlib import Path
import json

run_dir = Path(globals().get("RUN_DIR", globals().get("OUTPUT_DIR", ".")))
if not (run_dir / "logs").exists() and Path("runs").exists():
    run_candidates = sorted(Path("runs").glob("*"), key=lambda p: p.stat().st_mtime if p.exists() else 0)
    if run_candidates:
        run_dir = run_candidates[-1]
target_file = run_dir / "logs" / "target_resolution_all.json"
if not target_file.exists():
    target_file = run_dir / "logs" / "target_resolution.json"
if target_file.exists():
    target_data = json.loads(target_file.read_text(encoding="utf-8"))
    targets = target_data.get("targets") if isinstance(target_data, dict) and isinstance(target_data.get("targets"), list) else [target_data]
    for target in targets:
        if not isinstance(target, dict):
            continue
        print("Target:", target.get("target_label") or target.get("canonical_target") or target.get("target_id"))
        print("  primary_geometry_source:", target.get("primary_geometry_source"))
        print("  has_target_geometry_geojson:", bool(target.get("target_geometry_geojson") or target.get("primary_geometry_geojson")))
        print("  fallback_geometry_source:", target.get("fallback_geometry_source"))
        print("  fallback_geometry_bbox:", target.get("fallback_geometry_bbox") or target.get("nominatim_bbox"))
        print("  last_fallback_geometry_source:", target.get("last_fallback_geometry_source"))
        print("  effective_bbox:", target.get("effective_bbox") or target.get("bbox"))
        print("  geocoder point:", (target.get("chosen_candidate") or {}).get("lat"), (target.get("chosen_candidate") or {}).get("lon"))
else:
    print("No target-resolution log found yet:", target_file)

try:
    from camera_discovery.utils.geojson_viewer import write_embedded_camera_map
    if target_file.exists():
        map_path = write_embedded_camera_map(run_dir)
        status_path = run_dir / "logs" / "camera_map_status.json"
        status = json.loads(status_path.read_text(encoding="utf-8")) if status_path.exists() else {}
        print("Regenerated map:", map_path)
        print("Map primary geometry overlays:", status.get("target_primary_geometry_overlays"))
        print("Map fallback bbox overlays:", status.get("target_fallback_bbox_overlays"))
        print("Map last-fallback bbox overlays:", status.get("target_last_fallback_bbox_overlays"))
except Exception as exc:
    print("Map regeneration skipped/error:", repr(exc))


## Media validation dashboard and playlists
Inspect the new `media_validation_dashboard.json`, `playlists/`, and optional Google dorking summary artifacts when present.

In [ ]:
# Inspect media validation dashboard, playlists, and optional Google dorking counts
from pathlib import Path
import json, os

_run_dir_value = globals().get('OUTPUT_DIR') or globals().get('output_dir') or os.environ.get('CAMERA_DISCOVERY_OUTPUT_DIR') or 'runs/latest'
run_dir = Path(str(_run_dir_value))

dashboard_path = run_dir / 'media_validation_dashboard.json'
if dashboard_path.exists():
    print('media_validation_dashboard.json')
    print(json.dumps(json.loads(dashboard_path.read_text()), indent=2)[:4000])
else:
    print('media_validation_dashboard.json not found at', dashboard_path)

playlist_dir = run_dir / 'playlists'
if playlist_dir.exists():
    print('playlist artifacts:')
    for path in sorted(playlist_dir.glob('*')):
        print('-', path.relative_to(run_dir))
else:
    print('playlists/ not found at', playlist_dir)

dork_path = run_dir / 'logs' / 'google_dorking_summary.json'
if dork_path.exists():
    data = json.loads(dork_path.read_text())
    print('google_dorking:', {k: data.get(k) for k in ['enabled', 'queries_generated', 'results_seen', 'results_after_block_policy', 'promoted_source_leads', 'candidates_extracted']})
else:
    print('google_dorking summary not present')


## Passive intelligence summary

This notebook consumes source-code-generated passive intelligence artifacts. It does not patch repository source code. After a run completes, use this cell to inspect evidence-band counts, protocol labels, signature-family counts, and top evidence reasons from `media_validation_dashboard.json` and `logs/passive_intelligence_summary.json`.

In [ ]:

from pathlib import Path
import json


def _candidate_run_dirs():
    names = []
    for value_name in ["OUTPUT_DIR", "output_dir", "RUN_DIR", "run_dir", "ARTIFACT_DIR", "artifact_dir"]:
        value = globals().get(value_name)
        if value:
            names.append(Path(value))
    names.extend([Path("runs/latest"), Path("runs")])
    return names


def _find_passive_artifacts():
    for base in _candidate_run_dirs():
        if base.is_file():
            base = base.parent
        candidates = [base]
        if base.name == "runs" and base.exists():
            candidates.extend(sorted([p for p in base.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True))
        for run_dir in candidates:
            dashboard = run_dir / "media_validation_dashboard.json"
            passive = run_dir / "logs" / "passive_intelligence_summary.json"
            if dashboard.exists() or passive.exists():
                return run_dir, dashboard, passive
    return None, None, None

run_dir, dashboard_path, passive_path = _find_passive_artifacts()
if run_dir is None:
    print("Passive intelligence artifacts not found yet. Run the pipeline first, then rerun this cell.")
else:
    print(f"Passive intelligence artifacts: {run_dir}")
    dashboard = json.loads(dashboard_path.read_text(encoding="utf-8")) if dashboard_path and dashboard_path.exists() else {}
    passive = json.loads(passive_path.read_text(encoding="utf-8")) if passive_path and passive_path.exists() else dashboard.get("passive_intelligence", {})
    print("candidate_evidence_bands:", passive.get("candidate_evidence_bands", {}))
    print("protocol_label_counts:", passive.get("protocol_label_counts", {}))
    print("signature_family_counts:", passive.get("signature_family_counts", {}))
    print("top_camera_evidence_reasons:")
    for reason in passive.get("top_camera_evidence_reasons", [])[:10]:
        print("-", reason)
    print("\nDetailed artifacts:")
    for rel in [
        "logs/passive_intelligence_summary.json",
        "logs/source_row_evidence_summary.jsonl",
        "logs/candidate_evidence_summary.jsonl",
        "logs/candidate_priority_explanation.jsonl",
    ]:
        p = run_dir / rel
        print(f"- {rel}: {'present' if p.exists() else 'missing'}")
